# TRIBE v2 — Local Mac Backend

Runs the **brain-neuro** FastAPI backend locally on **Mac M3 Pro** (Apple Silicon).

> Uses **MPS** (Metal Performance Shaders) for GPU acceleration where supported, falls back to CPU.

**Steps:**
1. Run **Cell 1** — installs packages, then **restart the kernel manually** (Kernel → Restart).
2. After restart, run **Cells 2 → 6** in order.
3. Use `http://localhost:8000` as the backend URL in the frontend.

In [ ]:
# ── Cell 1 — Install & restart ──────────────────────────────────────
# Installs all required packages for running on Mac M3 Pro (Apple Silicon).

import subprocess, sys

def pip(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)

# uv is required at runtime — tribev2 uses `uvx whisperx` for transcription
pip(["uv"])

# tribev2 with plotting extra
pip(["tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"])

# Backend server + video download utilities
pip(["fastapi", "uvicorn[standard]", "yt-dlp", "pydantic", "nest_asyncio", "huggingface_hub"])

# ffmpeg — install via Homebrew if not already present
import shutil
if not shutil.which("ffmpeg"):
    print("Installing ffmpeg via Homebrew...")
    subprocess.run(["brew", "install", "ffmpeg"], check=True)
else:
    print(f"ffmpeg already installed: {shutil.which('ffmpeg')}")

print("\n✅ Installed. Please restart the kernel now (Kernel → Restart).")
print("   After restart, run Cell 2 onwards.")

In [ ]:
# ── Cell 2 — Verify environment (run after restart) ────────────────
import subprocess, sys, platform

# Check uv/uvx (needed by tribev2 for whisperx transcription)
try:
    subprocess.run(["uvx", "--version"], capture_output=True, check=True)
    print("uvx      : ✅")
except Exception:
    print("uvx      : ⚠ not found — run Cell 1 again and restart")

from tribev2.demo_utils import TribeModel
import tribev2, numpy as np, torch

# Detect Apple Silicon MPS
if torch.backends.mps.is_available():
    device_info = "Apple Silicon MPS ✅"
elif torch.cuda.is_available():
    device_info = torch.cuda.get_device_name(0)
else:
    device_info = "CPU-only"

print(f"tribev2  : {tribev2.__file__}")
print(f"numpy    : {np.__version__}")
print(f"torch    : {torch.__version__}")
print(f"Device   : {device_info}")
print(f"Chip     : {platform.processor() or platform.machine()}")
print(f"Python   : {sys.version.split()[0]}")
print("\n✅ All imports OK")

In [ ]:
# ── Cell 3 — Clone / update brain-neuro ─────────────────────────────
import os, subprocess

repo = os.path.expanduser('~/brain-neuro')
if os.path.exists(repo):
    subprocess.run(['git', '-C', repo, 'pull', '-q'], check=True)
    print('brain-neuro updated ✅')
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Sambhram1/brain-neuro.git', repo], check=True)
    print('brain-neuro cloned ✅')
os.chdir(repo)
print(f'cwd: {os.getcwd()}')

In [ ]:
# ── Cell 4 — HuggingFace login ──────────────────────────────────────
import os, huggingface_hub

hf_token = os.environ.get('HF_TOKEN') or input('Enter HuggingFace token (huggingface.co/settings/tokens): ')
huggingface_hub.login(token=hf_token, add_to_git_credential=False)
print('Logged in ✅ — model weights download on first /analyze call')

In [ ]:
# ── Cell 5 (Optional) — cookies.txt for Instagram/TikTok ───────────
# Skip this cell if you only use YouTube URLs.
# Place a cookies.txt file in the brain-neuro directory manually,
# or run this cell to check if one exists.
import os

cookies_path = os.path.join(os.getcwd(), 'cookies.txt')
if os.path.exists(cookies_path):
    print('cookies.txt ✅')
else:
    print('No cookies.txt found — YouTube only.')
    print(f'To enable Instagram/TikTok, place cookies.txt in: {os.getcwd()}')

In [ ]:
# ── Cell 6 — Start backend ──────────────────────────────────────────
import nest_asyncio, uvicorn, threading, time, os

repo = os.path.expanduser('~/brain-neuro')
os.chdir(repo)
nest_asyncio.apply()

# Launch FastAPI server in background thread
def run_server():
    uvicorn.run('backend:app', host='0.0.0.0', port=8000, log_level='info')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)

# Wait for server to be ready
import urllib.request
for attempt in range(15):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        print(f'\n{"═"*54}')
        print(f'  BACKEND URL: http://localhost:8000')
        print(f'{"═"*54}')
        print('→ Paste this URL into the frontend "Backend URL" field')
        break
    except Exception:
        if attempt == 14:
            raise RuntimeError('❌ FastAPI did not start — check for errors above')
        time.sleep(2)